# Import Libraries

In [24]:
import re
import nltk
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

# Dataset

In [4]:
df = pd.read_csv("/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")

In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    words = text.split()
    
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    
    return " ".join(words)

In [7]:
df["preprocessed_dataset"] = df["review"].apply(preprocess_text)

In [8]:
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

# PART ONE: Baseline Model (TF-IDF Only)

## TF-IDF Vectorization

In [10]:
vectorizer = TfidfVectorizer(max_features=5000)

X_tfidf = vectorizer.fit_transform(df["preprocessed_dataset"])
y = df["label"]

## Train-Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

## Baseline Model

In [12]:
baseline_model = LogisticRegression(max_iter=200)
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, baseline_preds))

Baseline Accuracy: 0.8893


# PART 2: Feature Engineering

## Text Length Feature

In [13]:
df["char_count"] = df["review"].apply(len)
df["word_count"] = df["review"].apply(lambda x: len(x.split()))

## POS Tag Features

In [14]:
def pos_features(text):
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)
    
    noun_count = sum(1 for word, tag in tags if tag.startswith("NN"))
    verb_count = sum(1 for word, tag in tags if tag.startswith("VB"))
    adj_count = sum(1 for word, tag in tags if tag.startswith("JJ"))
    
    return noun_count, verb_count, adj_count

In [17]:
df[["nouns", "verbs", "adjectives"]] = df["review"].apply(
    lambda x: pd.Series(pos_features(x))
)

## Combine Features

In [19]:
extra_features = df[["char_count", "word_count", "nouns", "verbs", "adjectives"]].values

## Convert TF-IDF + features into ONE matrix

In [20]:
X_combined = hstack([X_tfidf, extra_features])

## Train/Test Split Combined

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
)

## Train Improved Model

In [26]:
scaler = StandardScaler()
scaled_extra = scaler.fit_transform(extra_features)

X_combined = hstack([X_tfidf, scaled_extra])

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Improved Accuracy:", accuracy_score(y_test, preds))

Improved Accuracy: 0.8903


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Compare Results

In [27]:
print("Baseline:", accuracy_score(y_test, baseline_preds))
print("With Features:", accuracy_score(y_test, preds))

Baseline: 0.8893
With Features: 0.8903


# Feature Importance

In [28]:
feature_names = list(vectorizer.get_feature_names_out()) + [
    "char_count", "word_count", "nouns", "verbs", "adjectives"
]

coefficients = model.coef_[0]

top_features = sorted(zip(coefficients, feature_names), reverse=True)[:10]

top_features

[(np.float64(7.748644474798112), 'excellent'),
 (np.float64(6.931424055151677), 'great'),
 (np.float64(5.755632915300688), 'perfect'),
 (np.float64(5.533015107450243), 'wonderful'),
 (np.float64(5.411187238046238), 'amazing'),
 (np.float64(5.303308550319582), 'best'),
 (np.float64(4.93515321189486), 'favorite'),
 (np.float64(4.890487985953949), 'brilliant'),
 (np.float64(4.662396038625551), 'loved'),
 (np.float64(4.5416820698258205), 'enjoyed')]

# Key Concepts
Feature Engineering

Adding extra signals like:

- text length
- grammar
- structure

Why It Helps
- TF-IDF = words
- Features = behavior

Together = stronger model

# Limitations
- Some features may not improve performance
- Adds complexity
- Requires experimentation